# QuEra Analog Hamiltonian Simulation

Place neutral atoms in a geometry, drive them with time-dependent fields, and let the Rydberg Hamiltonian solve Maximum Independent Set -- no gate circuit in sight.

**Objectives:**
- Contrast *analog* quantum computation (QuEra Aquila) with the gate model of IonQ and IQM.
- Build a real `AnalogHamiltonianSimulation` program with `braket.ahs`: an `AtomArrangement` plus a time-dependent `DrivingField` (Rabi frequency, detuning, phase).
- Understand the *Rydberg blockade* and how atom geometry encodes a graph whose Maximum Independent Set the adiabatic sweep prepares.
- Inspect the program's register coordinates and field schedule locally, and cross-check the answer with a classical (numpy) MIS solve.
- Keep the actual Aquila submission behind `RUN_ON_AWS = False` with a cost note, per the project's local-first discipline.

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

## 0. Analog vs. gate model

IonQ and IQM are *gate-model* machines: you compose discrete gates into a circuit. QuEra **Aquila** is different. It is a 256-atom array of rubidium atoms held in optical tweezers, and it does **not** run gate circuits at all. Instead you:

1. **Place** the atoms in a 2D geometry (the *register*).
2. **Drive** them with global time-dependent fields -- a Rabi frequency $\Omega(t)$ that couples each atom's ground state $\ket{g}$ to a highly excited *Rydberg* state $\ket{r}$, and a detuning $\delta(t)$ that tilts the energy landscape.
3. **Let the system evolve** under the Rydberg Hamiltonian and read out which atoms ended up in $\ket{r}$.

The physics that makes this useful is the **Rydberg blockade**: two atoms closer than a *blockade radius* $R_b$ cannot both be excited to $\ket{r}$ at once -- exciting one shifts the other's energy out of resonance. So if you draw an edge between every pair of atoms within $R_b$, the set of simultaneously-excited atoms is always an **independent set** of that graph. Drive the system adiabatically and it relaxes toward the *largest* such set: the **Maximum Independent Set (MIS)**. The geometry *is* the program.

This notebook builds that program object for real with `braket.ahs`, inspects its structure, and checks the MIS classically. It runs locally in a venv with the real Amazon Braket SDK but **no AWS credentials** -- so we only ever *construct* and *inspect*; the single line that would submit to Aquila is guarded behind `RUN_ON_AWS = False`.

In [ ]:
# Setup: run this cell first.
# Requires the real amazon-braket-sdk (pip install -e '.[dev]' from the project root; see `make setup`).
# Importing braket.ahs and braket.aws is free and offline -- it does NOT contact AWS.
# We do NOT instantiate AwsDevice or call get_devices() at top level; any real call is guarded later.
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations, product

from braket.ahs import AtomArrangement, DrivingField, AnalogHamiltonianSimulation
from braket.timings.time_series import TimeSeries
from braket.aws import AwsDevice  # imported only to show the submission API; never called unless RUN_ON_AWS

# Aquila's device ARN is just a string until you hand it to AwsDevice(...).
AQUILA_ARN = "arn:aws:braket:us-east-1::device/qpu/quera/Aquila"

# Cost discipline (project rule): keep the real submission OFF by default.
RUN_ON_AWS = False

print("braket.ahs imported; we will construct an AHS program locally (no AWS calls).")
print(f"Aquila device ARN: {AQUILA_ARN}")
print(f"RUN_ON_AWS = {RUN_ON_AWS}")

## 1. The atom geometry (register)

We lay out **five atoms on a regular pentagon ring**. Aquila works in physical units: coordinates are in **meters**, and realistic atom spacings are a few micrometers. With the right spacing, each atom is within the blockade radius of its two ring neighbors but *not* its two non-neighbors -- so the blockade graph is a 5-cycle ($C_5$), a small problem whose MIS we can verify by hand.

We place the ring with radius `6 um` and read off the nearest-neighbor and next-nearest-neighbor distances; the blockade radius will sit between them.

In [ ]:
N = 5
ring_radius = 6.0e-6  # meters (6 micrometers)

# Evenly spaced points on a circle; the +pi/2 just puts atom 0 at the top.
angles = np.array([2 * np.pi * k / N + np.pi / 2 for k in range(N)])
coords = np.stack([ring_radius * np.cos(angles), ring_radius * np.sin(angles)], axis=1)

print("Atom coordinates (micrometers):")
for k, (x, y) in enumerate(coords):
    print(f"  atom {k}: ({x * 1e6:6.3f}, {y * 1e6:6.3f})")

nn = np.linalg.norm(coords[0] - coords[1])   # adjacent on the ring
nnn = np.linalg.norm(coords[0] - coords[2])  # skip one
print(f"\nnearest-neighbor spacing:      {nn * 1e6:6.3f} um")
print(f"next-nearest-neighbor spacing: {nnn * 1e6:6.3f} um")

# Sanity: on a pentagon, adjacent atoms are strictly closer than skip-one atoms.
assert nn < nnn, "ring geometry should make neighbors closer than non-neighbors"

## 2. Blockade graph and the classical MIS

Choose the blockade radius $R_b$ to sit **between** the nearest- and next-nearest spacings. Then exactly the five ring edges are "blockaded": each pair of adjacent atoms cannot both be excited. That is the 5-cycle graph $C_5$.

We brute-force its Maximum Independent Set in numpy (only $2^5 = 32$ bitstrings) so we have a ground-truth answer to compare against the analog program. For a 5-cycle the MIS has size 2 -- you can pick at most every other atom around an odd ring.

In [ ]:
blockade_radius = 0.5 * (nn + nnn)  # between neighbor and non-neighbor spacing
print(f"blockade radius R_b = {blockade_radius * 1e6:.3f} um")

# Edge if the pair is within R_b: they cannot both be Rydberg-excited.
edges = []
for i, j in combinations(range(N), 2):
    if np.linalg.norm(coords[i] - coords[j]) <= blockade_radius:
        edges.append((i, j))
print("blockade edges (pairs that cannot both be |r>):", edges)
assert len(edges) == 5, "the pentagon should yield exactly the 5 ring edges (a 5-cycle)"

# Brute-force Maximum Independent Set over all 2^N selections.
best_set, best_size = [], -1
for bits in product([0, 1], repeat=N):
    independent = all(not (bits[i] and bits[j]) for (i, j) in edges)
    if independent:
        chosen = [k for k in range(N) if bits[k]]
        if len(chosen) > best_size:
            best_size, best_set = len(chosen), chosen

print(f"classical MIS size = {best_size}, e.g. atoms {best_set}")
# The maximum independent set of an odd cycle C_5 is floor(5/2) = 2.
assert best_size == 2, "MIS of a 5-cycle must be 2"

## 3. Visualize the register and its blockade graph

Atoms in the MIS are highlighted; grey lines are blockade edges. The analog machine's job is to find this set physically -- by evolving atoms toward the highest-occupancy independent configuration.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for (i, j) in edges:
    ax.plot(
        [coords[i, 0] * 1e6, coords[j, 0] * 1e6],
        [coords[i, 1] * 1e6, coords[j, 1] * 1e6],
        "-", color="0.75", zorder=1,
    )
in_mis = set(best_set)
for k, (x, y) in enumerate(coords):
    ax.scatter(
        x * 1e6, y * 1e6, s=420, zorder=3, edgecolors="k",
        c="#d1495b" if k in in_mis else "#3a6ea5",
    )
    ax.annotate(str(k), (x * 1e6, y * 1e6), ha="center", va="center", color="w", zorder=4)
ax.set_xlabel("x (um)")
ax.set_ylabel("y (um)")
ax.set_title("Pentagon register: red = a Maximum Independent Set")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 4. The time-dependent driving field

The global drive has three knobs, each a `TimeSeries` over the protocol duration `t_max`:

- **Amplitude** $\Omega(t)$ -- the Rabi frequency coupling $\ket{g}\leftrightarrow\ket{r}$ (rad/s). We ramp it up, hold, and ramp down so the atoms start and end idle in $\ket{g}$.
- **Detuning** $\delta(t)$ -- swept from **negative to positive**. This is the adiabatic trick: at large negative $\delta$ the ground state has *no* excitations; as $\delta$ crosses positive, exciting atoms becomes favorable, but the blockade caps how many neighbors can light up at once. Sweep slowly and the system tracks the instantaneous ground state straight into a Maximum Independent Set.
- **Phase** $\phi(t)$ -- held at 0 here (we do not need a phase profile for this protocol).

Values below are in Aquila-realistic ranges (a few Mrad/s, microsecond timescales). We print the full schedule so you can see real local output.

In [ ]:
t_max = 4.0e-6           # total evolution time: 4 microseconds
Omega_max = 1.5e7        # peak Rabi frequency ~15 Mrad/s
delta_start, delta_end = -2.0e7, 2.0e7  # detuning swept negative -> positive (rad/s)
ramp = 0.25e-6           # amplitude ramp up/down time

# Amplitude: 0 -> Omega_max (ramp up), hold, -> 0 (ramp down). Atoms idle in |g> at both ends.
amplitude = (
    TimeSeries()
    .put(0.0, 0.0)
    .put(ramp, Omega_max)
    .put(t_max - ramp, Omega_max)
    .put(t_max, 0.0)
)
# Detuning: linear sweep from negative to positive -- the adiabatic ramp that prepares the MIS.
detuning = TimeSeries().put(0.0, delta_start).put(t_max, delta_end)
# Phase: flat at 0.
phase = TimeSeries().put(0.0, 0.0).put(t_max, 0.0)

print("Amplitude (Rabi frequency) schedule  [time -> Omega]:")
for t, v in zip(amplitude.times(), amplitude.values()):
    print(f"  t = {t * 1e6:5.2f} us   Omega = {v / 1e6:6.2f} Mrad/s")
print("Detuning schedule  [time -> delta]:")
for t, v in zip(detuning.times(), detuning.values()):
    print(f"  t = {t * 1e6:5.2f} us   delta = {v / 1e6:6.2f} Mrad/s")

# The protocol must start and end with the drive off (atoms in |g>)...
assert amplitude.values()[0] == 0.0 and amplitude.values()[-1] == 0.0
# ...and the detuning must cross from negative to positive for the adiabatic sweep.
assert detuning.values()[0] < 0 < detuning.values()[-1]

## 5. Assemble the `AnalogHamiltonianSimulation` program

Now we build the real SDK objects: an `AtomArrangement` (via `.add(...)` for each atom), a `DrivingField` from the three time series, and the `AnalogHamiltonianSimulation(register=..., hamiltonian=...)`. This is exactly what you would submit to Aquila -- but constructing and inspecting it touches no network. We print the register coordinates read back **from the program object** and confirm it carries one Hamiltonian term (the single global drive).

In [ ]:
register = AtomArrangement()
for (x, y) in coords:
    register.add((float(x), float(y)))  # coordinates in meters

drive = DrivingField(amplitude=amplitude, phase=phase, detuning=detuning)
ahs_program = AnalogHamiltonianSimulation(register=register, hamiltonian=drive)

print(f"register atoms:      {len(ahs_program.register)}")
print(f"hamiltonian terms:   {len(ahs_program.hamiltonian.terms)}  (one global driving field)")
print("\nregister site coordinates read back from the program object:")
for k, site in enumerate(ahs_program.register):
    cx, cy = site.coordinate
    print(f"  site {k}: ({float(cx) * 1e6:6.3f}, {float(cy) * 1e6:6.3f}) um   [{site.site_type}]")

assert len(ahs_program.register) == N
assert len(ahs_program.hamiltonian.terms) == 1  # exactly the one DrivingField
print("\nProgram constructed locally and inspected -- no AWS contact.")

## 6. Submitting to Aquila (guarded)

Running `ahs_program` on Aquila is a **single line**: `AwsDevice(AQUILA_ARN).run(ahs_program, shots=...)`. We keep it behind `RUN_ON_AWS`, which is `False`, so this notebook never instantiates `AwsDevice`, never queries device properties, and never submits a task.

**Cost note (per the project's cost rule).** QuEra Aquila is metered like other QPUs: roughly **$0.30 per task** plus **$0.01 per shot**. A typical 100-shot run is about `$0.30 + 100 x $0.01 = $1.30`, before any queue wait. Aquila is also region- and window-limited (availability windows, max ~256 atoms, geometry constraints checked against `device.properties`). Only flip `RUN_ON_AWS = True` with valid AWS credentials, after validating the geometry and schedule -- which is exactly what we did above for free.

In [ ]:
if RUN_ON_AWS:
    # COST: ~$0.30 per task + $0.01 per shot. Requires AWS credentials and Aquila availability.
    device = AwsDevice(AQUILA_ARN)
    n_shots = 100
    # (Optionally validate against device.properties / paradigm constraints before submitting.)
    task = device.run(ahs_program, shots=n_shots)
    result = task.result()
    # Each shot reports per-site status (empty / |g> ground / |r> Rydberg);
    # Rydberg sites approximate an independent set, and the most common one ~ the MIS.
    print(result.measurements[:3])
else:
    print("RUN_ON_AWS is False -- no AwsDevice created, no task submitted, no charge incurred.")
    print("The program above is fully built and validated locally; the classical MIS is the")
    print(f"expected answer: size {best_size} (e.g. atoms {best_set}).")

## Exercises

Five exercises that push on the geometry-is-the-program idea. Each has tiered
hints -- expand only what you need -- and a check cell that tells you when you
have it. Everything runs locally: no exercise needs AWS credentials, and the
one that touches the submission path keeps it behind the same `RUN_ON_AWS`
guard as Section 6.

### Exercise 1 — A hexagon register (C6)

The pentagon was an odd cycle; make it even. Rebuild the ring with six atoms
at the same `ring_radius`, pick a blockade radius that sits between the new
neighbor and skip-one spacings, and brute-force the MIS of the resulting
blockade graph. An even cycle's MIS is N/2, so you should land on 3.

Define `hex_coords` (six (x, y) positions in meters), `hex_edges` (the
blockade edge list over atoms 0-5), and `hex_mis_size` (the brute-forced
Maximum Independent Set size).

<details><summary>Hint 1 — nudge</summary>

On a ring only two distances matter: neighbor spacing and skip-one spacing.
Both change when the atom count changes at fixed radius — where must the
blockade radius sit relative to those two for the graph to be exactly the
ring?

</details>
<details><summary>Hint 2 — approach</summary>

Repeat Section 1's construction with six angles instead of five, measure the
neighbor and skip-one distances with `np.linalg.norm`, choose a radius between
them, then rebuild the edge list with `combinations` and rerun the
2^6-bitstring brute force from Section 2 on the new edges.

</details>

In [ ]:
# Exercise 1: Rebuild the register as a 6-atom hexagon ring and brute-force its MIS.
# Define: hex_coords -- six (x, y) positions in meters; hex_edges -- the
# blockade edge list; hex_mis_size -- the brute-forced MIS size.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    _hx = np.asarray(hex_coords, dtype=float)
    assert _hx.shape == (6, 2), "hex_coords should hold six (x, y) positions"
    _hd = [np.linalg.norm(_hx[i] - _hx[j]) for i, j in combinations(range(6), 2)]
    assert 1e-7 < min(_hd) < 1e-4, (
        "keep coordinates in meters at micrometer scale, like the pentagon"
    )
    _deg = {k: 0 for k in range(6)}
    for i, j in hex_edges:
        _deg[int(i)] += 1
        _deg[int(j)] += 1
    assert len(hex_edges) == 6 and all(d == 2 for d in _deg.values()), (
        "a hexagon ring should blockade exactly its six neighbor pairs"
    )
    assert hex_mis_size == 3, "an even cycle C_6 has an MIS of N/2 = 3"

### Exercise 2 — Tune the blockade

The drive fixes the blockade radius; you control the geometry. Keep the
notebook's `blockade_radius` unchanged and shrink the pentagon (a ring radius
around 4.5 um) until skip-one pairs also fall inside it, then see what that
does to the blockade graph and its MIS.

Define `dense_coords` (the five shrunken (x, y) positions in meters),
`dense_edges` (edges under the unchanged `blockade_radius`), and
`dense_mis_size`.

<details><summary>Hint 1 — nudge</summary>

An edge is nothing but a distance below threshold. If every pairwise distance
in the register drops below the blockade radius, what graph do five atoms
form — and how many of them can be Rydberg-excited at once?

</details>
<details><summary>Hint 2 — approach</summary>

Rebuild the pentagon coordinates exactly as Section 1 did but with the smaller
ring radius, leave `blockade_radius` alone, and rerun the `combinations` edge
scan plus the 32-bitstring brute force from Section 2 on the new edge list.

</details>

In [ ]:
# Exercise 2: Shrink the pentagon until skip-one pairs also blockade.
# Define: dense_coords -- five (x, y) positions in meters; dense_edges -- edges
# under the unchanged blockade_radius; dense_mis_size -- the new MIS size.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    _dc = np.asarray(dense_coords, dtype=float)
    assert _dc.shape == (5, 2), "dense_coords should hold five (x, y) positions"
    assert all(
        np.linalg.norm(_dc[i] - _dc[j]) <= blockade_radius
        for i, j in combinations(range(5), 2)
    ), "shrink the ring until even skip-one pairs sit inside the unchanged R_b"
    assert len(dense_edges) == 10, "once every pair blockades, all ten pairs are edges"
    assert dense_mis_size == 1, (
        "when every atom blockades every other, only one atom can be excited"
    )

### Exercise 3 — A hub atom in the center

Add a sixth atom at the center of the original pentagon. The center sits one
`ring_radius` (6 um) from every ring atom — inside the blockade radius — so it
is blockaded with all five. Verify numerically that such a hub can never
appear in a maximum independent set.

Define `hub_coords` (six (x, y) positions: the pentagon plus the center),
`hub_edges`, `hub_mis_size`, and `hub_in_any_mis` — True only if some
maximum-size independent set contains the center atom.

<details><summary>Hint 1 — nudge</summary>

Compare the center-to-ring distance with the blockade radius. A vertex
adjacent to every other vertex forces any independent set containing it to
stop at size one — how does that compare to the ring's own MIS?

</details>
<details><summary>Hint 2 — approach</summary>

Stack the center point onto `coords` with `np.vstack`, recompute the edge list
over `range(6)`, and brute-force 2^6 bitstrings as before — but this time,
while enumerating, also record whether any independent set of maximum size has
the center atom's bit set.

</details>

In [ ]:
# Exercise 3: Add a hub atom at the pentagon's center and test its MIS role.
# Define: hub_coords -- six (x, y) positions (pentagon + center); hub_edges;
# hub_mis_size; hub_in_any_mis -- True only if a maximum-size independent set
# contains the center atom.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    _hc = np.asarray(hub_coords, dtype=float)
    assert _hc.shape == (6, 2), "hub_coords should hold the five ring atoms plus the center"
    _deg = {k: 0 for k in range(6)}
    for i, j in hub_edges:
        _deg[int(i)] += 1
        _deg[int(j)] += 1
    assert max(_deg.values()) == 5, "one atom (the hub) should blockade with all five others"
    assert len(hub_edges) == 10, "expect the five ring edges plus five spokes to the hub"
    assert hub_mis_size == 2, "a fully blockaded hub cannot grow the ring's MIS beyond 2"
    assert not hub_in_any_mis, (
        "an atom blockaded with everything cannot sit in any maximum independent set"
    )

### Exercise 4 — A slower adiabatic sweep

Adiabatic theory says a slower sweep tracks the instantaneous ground state
better, which on hardware means a higher MIS success probability. Double the
protocol duration while keeping the same start and end detuning, and quantify
what that does to the sweep rate.

Define `slow_t_max` (twice the notebook's `t_max`) and `slow_detuning` (a
`TimeSeries` sweeping `delta_start` to `delta_end` over the longer window).
Print the new schedule and the old and new sweep rates in rad/s^2.

<details><summary>Hint 1 — nudge</summary>

What matters for adiabaticity is the detuning's rate of change, not its
endpoints. If the swept span is fixed and the time doubles, what happens to
that rate — and why do slower crossings of the gap minimum cause fewer
diabatic (Landau-Zener) errors?

</details>
<details><summary>Hint 2 — approach</summary>

Mirror Section 4's detuning construction: a fresh `TimeSeries` with one `.put`
at time zero and one at the doubled end time, reusing the notebook's
`delta_start` and `delta_end` unchanged. The sweep rate is the swept span
divided by the duration.

</details>

In [ ]:
# Exercise 4: Double the protocol duration at the same start/end detuning.
# Define: slow_t_max -- twice t_max; slow_detuning -- a TimeSeries sweeping
# delta_start to delta_end over the longer window.

# TODO: your code here

In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    assert np.isclose(slow_t_max, 2 * t_max), "slow_t_max should be exactly twice t_max"
    _st = [float(t) for t in slow_detuning.times()]
    _sv = [float(v) for v in slow_detuning.values()]
    assert np.isclose(_st[0], 0.0) and np.isclose(_st[-1], slow_t_max), (
        "the sweep should span time 0 to slow_t_max"
    )
    assert np.isclose(_sv[0], delta_start) and np.isclose(_sv[-1], delta_end), (
        "keep the same start and end detuning as the original sweep"
    )
    _rate = (_sv[-1] - _sv[0]) / (_st[-1] - _st[0])
    assert np.isclose(_rate, 0.5 * (delta_end - delta_start) / t_max), (
        "twice the time over the same span should halve the sweep rate"
    )

### Exercise 5 — Pre-flight register validation

Every metered submission should be preceded by a free validation pass.
Aquila's published paradigm limits include at most 256 atoms and a minimum
atom spacing of 4 um. Write the validator and run it on the notebook's
register.

Define `validate_register(candidate, max_atoms, min_spacing)` — a function
returning True only when the `AtomArrangement` `candidate` has at most
`max_atoms` sites and no pair of sites closer than `min_spacing` — and
`register_ok`, its verdict on the notebook's pentagon `register` under those
published limits. (With credentials and `RUN_ON_AWS = True` you could read
the live numbers from `AwsDevice(AQUILA_ARN).properties.paradigm` instead —
reading properties is free, only `.run()` is metered — but keep that path
guarded exactly like Section 6.)

<details><summary>Hint 1 — nudge</summary>

An `AtomArrangement` is iterable and each site carries a `.coordinate`. The
function has exactly two ways to say no: one about how many sites there are,
one about how close the closest pair sits.

</details>
<details><summary>Hint 2 — approach</summary>

Collect the site coordinates into a list of (x, y) floats, compare the site
count against `max_atoms`, then scan every pair with `combinations` and a
Euclidean norm, failing the moment a pair is closer than `min_spacing`.

</details>

In [ ]:
# Exercise 5: Validate the register against Aquila's published paradigm limits.
# Define: validate_register(candidate, max_atoms, min_spacing) -- bool; and
# register_ok -- its verdict on the notebook's pentagon register.

# TODO: your code here

In [ ]:
# Check Exercise 5 -- run after your attempt.
from lib.grading import check

with check("Exercise 5"):
    assert callable(validate_register), "validate_register should be a function"
    assert validate_register(register, 256, 4.0e-6), (
        "the pentagon register sits well inside Aquila's published limits"
    )
    assert register_ok, "register_ok should record the pentagon register's verdict"
    _tight = AtomArrangement()
    _tight.add((0.0, 0.0))
    _tight.add((1.0e-6, 0.0))
    assert not validate_register(_tight, 256, 4.0e-6), (
        "two atoms 1 um apart violate a 4 um minimum spacing"
    )
    _crowd = AtomArrangement()
    for _k in range(9):
        _crowd.add((_k * 10.0e-6, 0.0))
    assert not validate_register(_crowd, 8, 4.0e-6), (
        "nine well-spaced atoms should still fail a max-atom limit of eight"
    )

## Summary

- **Aquila is analog, not gate-model.** You specify a *geometry* of neutral atoms and a *time-dependent drive* (Rabi frequency $\Omega(t)$, detuning $\delta(t)$, phase $\phi(t)$), then evolve under the Rydberg Hamiltonian -- no circuit.
- **The Rydberg blockade encodes a graph.** Atoms within the blockade radius $R_b$ cannot both reach $\ket{r}$, so excited atoms form an *independent set*; an adiabatic detuning sweep (negative to positive) drives the array toward a **Maximum Independent Set**.
- **We built the real program with `braket.ahs`** -- `AtomArrangement.add(...)`, `DrivingField`, `AnalogHamiltonianSimulation(register=..., hamiltonian=...)` -- and inspected its register coordinates and one Hamiltonian term locally, with **no AWS calls**.
- **A 5-atom pentagon is the 5-cycle $C_5$,** whose MIS (size 2) we verified classically in numpy as ground truth for the analog answer.
- **Cost discipline held:** the only line that would spend money, `AwsDevice(AQUILA_ARN).run(ahs_program, shots=...)`, stayed behind `RUN_ON_AWS = False` with a `$0.30/task + $0.01/shot` estimate -- local-first, exactly the workflow the module preaches.

**Next:** [`05-simulator-comparison.ipynb`](05-simulator-comparison.ipynb) -- run the same gate circuit across the simulator ladder (Local, SV1, DM1, TN1) and weigh accuracy, runtime, and cost.